In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
import duckdb


dbox = "C:/Users/eunic/Dropbox/sa_fires"
int_path = f"{dbox}/proj_bureaucrats_farms/data_output/intermediate"
myneta_path = f"{dbox}/data/input/my_neta"

In [2]:
# Importing Information
final_df = pd.read_csv(f'{int_path}/_ac_covs_myneta.csv')
election_info = pd.read_csv(f"{myneta_path}/ac_india_elec_yr_clean.csv").rename(columns = {'year' : 'election_year'})


In [3]:

#Year and month data generation
months = pd.DataFrame({'month' : np.arange(1, 13)})
months['count']  = 1
yrs = pd.DataFrame({'year' : np.arange(2000, 2030)})
yrs['count'] = 1
months = months.merge(yrs).sort_values(['year', 'month'])

# ACs 
ac_by_year = final_df[['state', 'ac_uq_id']].drop_duplicates()
ac_by_year['count']=1
ac_by_year = ac_by_year.merge(months).drop('count', axis = 1)


In [4]:

ac_panel = ac_by_year.merge( election_info, left_on = ['state', 'year', 'month'], 
                 right_on = [ 'state', 'year_take', 'month_take'], how = 'left')
ac_panel = ac_panel.sort_values(['state', 'ac_uq_id', 'year', 'month'])
ac_panel[ac_panel.columns.difference(['state', 'ac_uq_id'])] = (
    ac_panel.groupby(['state', 'ac_uq_id'])[ac_panel.columns.difference(['state', 'ac_uq_id'])].ffill()
)      
# Create a numeric year-month for easy comparison
ac_panel["ym"] = ac_panel["year"] * 12 + ac_panel["month"]
ac_panel["ym_take"] = ac_panel["year_take"] * 12 + ac_panel["month_take"]


In [5]:
# Keep rows where year-month >= year_take/month_take AND within 60 months
ac_panel = ac_panel[
    (ac_panel["ym"] >= ac_panel["ym_take"]) & 
    (ac_panel["ym"] < ac_panel["ym_take"] + 60)
]
# Clean up helper columns
ac_panel = ac_panel.drop(columns=["ym", "ym_take"])
ac_panel = ac_panel.drop(['count', 'day_take', 'day_counting', 'month_counting', 'year_counting', 'nseats'], axis = 1)
ac_panel["ym"] = ac_panel["year"] * 12 + ac_panel["month"]
ac_panel["ym_take"] = ac_panel["year_take"] * 12 + ac_panel["month_take"]
ac_panel["yeargov"] = ((ac_panel["ym"] - ac_panel["ym_take"]) // 12) + 1

In [ ]:
panel_acs_elec_covs = ac_panel.merge(final_df, on = ['state', 'ac_uq_id', 'election_year'], how = 'left')
fixed_cols = ['acpost08ID', 'ASSEMBLY', 'ASSEMBLY_1', 'DISTRICT', 'PARLIAMENT', 'P_NAME', 'STATE_UT']
# Fill AC-identifying attributes WITHIN each AC (they are constant per ac_uq_id),
# not globally. Filling with a single global value would stamp one AC's identity
# onto every AC that had a missing election-year match.
panel_acs_elec_covs[fixed_cols] = (
    panel_acs_elec_covs
        .groupby('ac_uq_id')[fixed_cols]
        .transform(lambda s: s.ffill().bfill())
)



C:\Users\eunic\AppData\Local\Temp\ipykernel_34048\3664636270.py:12: InvalidColumnName: 
Not all pandas column names were valid Stata variable names.
The following replacements have been made:

    dependent_1_owns_agricultural_assets   ->   dependent_1_owns_agricultural_as
    dependent_2_owns_agricultural_assets   ->   dependent_2_owns_agricultural_as
    dependent_3_owns_agricultural_assets   ->   dependent_3_owns_agricultural_as

If this is not what you expect, please make sure you have Stata-compliant
column names in your DataFrame (strings only, max 32 characters, only
alphanumerics and underscores, no Stata reserved words)

  panel_acs_elec_covs.to_stata(f'{int_path}/panel_data_election_year.dta')


In [9]:
# ------------------------------------------------------------------
# Output paths
# ------------------------------------------------------------------
output_dir = Path(int_path)
output_dir.mkdir(parents=True, exist_ok=True)

parquet_path = output_dir / "panel_data_election_year.parquet"
duckdb_path = output_dir / "panel_data_election_year.duckdb"
stata_path = output_dir / "panel_data_election_year.dta"

# ------------------------------------------------------------------
# Save as Parquet
# ------------------------------------------------------------------
panel_acs_elec_covs.to_parquet(
    parquet_path,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

# ------------------------------------------------------------------
# Save as DuckDB
# ------------------------------------------------------------------
with duckdb.connect(str(duckdb_path)) as con:
    con.register("panel_df", panel_acs_elec_covs)

    con.execute("""
        CREATE OR REPLACE TABLE panel_acs_elec_covs AS
        SELECT *
        FROM panel_df
    """)

    con.unregister("panel_df")

# ------------------------------------------------------------------
# Optional: save as Stata
# ------------------------------------------------------------------
panel_acs_elec_covs.to_stata(
    stata_path,
    write_index=False,
    version=118,
)

print(f"Parquet saved to: {parquet_path}")
print(f"DuckDB saved to:  {duckdb_path}")
print(f"Stata saved to:   {stata_path}")

C:\Users\eunic\AppData\Local\Temp\ipykernel_34048\1422635461.py:38: InvalidColumnName: 
Not all pandas column names were valid Stata variable names.
The following replacements have been made:

    dependent_1_owns_agricultural_assets   ->   dependent_1_owns_agricultural_as
    dependent_2_owns_agricultural_assets   ->   dependent_2_owns_agricultural_as
    dependent_3_owns_agricultural_assets   ->   dependent_3_owns_agricultural_as

If this is not what you expect, please make sure you have Stata-compliant
column names in your DataFrame (strings only, max 32 characters, only
alphanumerics and underscores, no Stata reserved words)

  panel_acs_elec_covs.to_stata(


Parquet saved to: C:\Users\eunic\Dropbox\sa_fires\proj_bureaucrats_farms\data_output\intermediate\panel_data_election_year.parquet
DuckDB saved to:  C:\Users\eunic\Dropbox\sa_fires\proj_bureaucrats_farms\data_output\intermediate\panel_data_election_year.duckdb
Stata saved to:   C:\Users\eunic\Dropbox\sa_fires\proj_bureaucrats_farms\data_output\intermediate\panel_data_election_year.dta
